# **1. upload file**

In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

df = pd.read_csv("StudentPerformanceFactors.csv")

print("=== DATA AWAL ===")
print(df.head())

Saving StudentPerformanceFactors.csv to StudentPerformanceFactors.csv
=== DATA AWAL ===
   Hours_Studied  Attendance Parental_Involvement Access_to_Resources  \
0             23          84                  Low                High   
1             19          64                  Low              Medium   
2             24          98               Medium              Medium   
3             29          89                  Low              Medium   
4             19          92               Medium              Medium   

  Extracurricular_Activities  Sleep_Hours  Previous_Scores Motivation_Level  \
0                         No            7               73              Low   
1                         No            8               59              Low   
2                        Yes            7               91           Medium   
3                        Yes            8               98           Medium   
4                        Yes            6               65           Medium   

# **2. Data Cleaning**

In [ ]:
# rapikan nama kolom
df.columns = df.columns.str.strip().str.lower()

print("\n=== NAMA KOLOM ===")
print(df.columns)

# cek missing value
print("\n=== MISSING VALUE ===")
print(df.isnull().sum())

# hapus data kosong
df = df.dropna()

print("\n=== DATA SETELAH CLEANING ===")
print(df.head())


=== NAMA KOLOM ===
Index(['hours_studied', 'attendance', 'parental_involvement',
       'access_to_resources', 'extracurricular_activities', 'sleep_hours',
       'previous_scores', 'motivation_level', 'internet_access',
       'tutoring_sessions', 'family_income', 'teacher_quality', 'school_type',
       'peer_influence', 'physical_activity', 'learning_disabilities',
       'parental_education_level', 'distance_from_home', 'gender',
       'exam_score'],
      dtype='object')

=== MISSING VALUE ===
hours_studied                  0
attendance                     0
parental_involvement           0
access_to_resources            0
extracurricular_activities     0
sleep_hours                    0
previous_scores                0
motivation_level               0
internet_access                0
tutoring_sessions              0
family_income                  0
teacher_quality               78
school_type                    0
peer_influence                 0
physical_activity              0

# **3. Encoding Kategori**

In [ ]:
df = df.copy()

# 1. Hours Studied
df.loc[:, 'hours_cat'] = df['hours_studied'].apply(
    lambda x: 'low' if x > 7 else ('medium' if 5 <= x <= 7 else 'high')
)

# 2. Previous Scores
df.loc[:, 'score_cat'] = df['previous_scores'].apply(
    lambda x: 'low' if x >= 75 else ('medium' if 60 <= x <= 74 else 'high')
)

# 3. Sleep Hours
df.loc[:, 'sleep_cat'] = df['sleep_hours'].apply(
    lambda x: 'low' if x > 8 else ('medium' if 6 <= x <= 8 else 'high')
)

# 4. Motivation Level
df.loc[:, 'motivation_cat'] = df['motivation_level'].str.lower()

# 5. Parental Involvement
df.loc[:, 'parental_cat'] = df['parental_involvement'].str.lower()

# 6. Access to Resources
df.loc[:, 'resource_cat'] = df['access_to_resources'].str.lower()

# 7. Tutoring Session
df.loc[:, 'tutoring_cat'] = df['tutoring_sessions'].apply(
    lambda x: 'low' if str(x).lower() == 'rutin'
    else ('medium' if str(x).lower() == 'jarang' else 'high')
)

# 8. Learning Disabilities
df.loc[:, 'disability_cat'] = df['learning_disabilities'].apply(
    lambda x: 'low' if str(x).lower() == 'tidak ada'
    else ('medium' if str(x).lower() == 'ringan' else 'high')
)

print("\n=== HASIL KATEGORI (8 VARIABEL) ===")
print(df[['hours_cat','score_cat','sleep_cat',
          'motivation_cat','parental_cat','resource_cat',
          'tutoring_cat','disability_cat']].head())


=== HASIL KATEGORI (8 VARIABEL) ===
  hours_cat score_cat sleep_cat motivation_cat parental_cat resource_cat  \
0       low    medium    medium            low          low         high   
1       low      high    medium            low          low       medium   
2       low       low    medium         medium       medium       medium   
3       low       low    medium         medium          low       medium   
4       low    medium    medium         medium       medium       medium   

  tutoring_cat disability_cat  
0         high           high  
1         high           high  
2         high           high  
3         high           high  
4         high           high  


# **4. Pre-Processing Modul 1 (Risiko Akademik)**

In [ ]:
df = df.copy()

def risk_analysis(row):
    categories = [
        row['hours_cat'],
        row['score_cat'],
        row['sleep_cat'],
        row['motivation_cat'],
        row['parental_cat'],
        row['resource_cat'],
        row['tutoring_cat'],
        row['disability_cat']
    ]

    high = categories.count('high')
    medium = categories.count('medium')
    low = categories.count('low')

    # =========================
    # RULE SESUAI TABEL
    # =========================

    # 1. ≥ 5 High
    if high >= 5:
        return 'High Risk'

    # 2. ≥ 3 High dan sisanya Medium
    elif high >= 3 and low == 0:
        return 'High Risk'

    # 3. ≥ 3 High + ≥ 2 Medium + ≥ 1 Low
    elif high >= 3 and medium >= 2 and low >= 1:
        return 'High Risk'

    # 4. 1 High + ≥ 2 Medium
    elif high == 1 and medium >= 2:
        return 'Medium Risk'

    # 5. ≥ 2 High dan tidak ada Low
    elif high >= 2 and low == 0:
        return 'Medium Risk'

    # 6. Mayoritas Medium
    elif medium > high and medium > low:
        return 'Medium Risk'

    # 7. Semua Low
    elif low == 8:
        return 'Low Risk'

    # 8. Mayoritas Low
    elif low > high and low > medium:
        return 'Low Risk'

    # default
    return 'Medium Risk'


# APPLY TANPA WARNING
df.loc[:, 'risk_level'] = df.apply(risk_analysis, axis=1)


# OUTPUT LENGKAP
print("\n=== HASIL ANALISA RISIKO ===")
print(df[[
    'hours_cat',
    'score_cat',
    'sleep_cat',
    'motivation_cat',
    'parental_cat',
    'resource_cat',
    'tutoring_cat',
    'disability_cat',
    'risk_level'
]].head())


=== HASIL ANALISA RISIKO ===
  hours_cat score_cat sleep_cat motivation_cat parental_cat resource_cat  \
0       low    medium    medium            low          low         high   
1       low      high    medium            low          low       medium   
2       low       low    medium         medium       medium       medium   
3       low       low    medium         medium          low       medium   
4       low    medium    medium         medium       medium       medium   

  tutoring_cat disability_cat   risk_level  
0         high           high    High Risk  
1         high           high    High Risk  
2         high           high  Medium Risk  
3         high           high  Medium Risk  
4         high           high  Medium Risk  


# **5. Pre-Processing Modul 2 (Rekomendasi Teknik Belajar)**

In [ ]:
# pastikan aman dari warning
df = df.copy()

def rekomendasi(row):

    # RULE 1
    if (row['sleep_cat'] == 'medium' and
        row['hours_cat'] == 'low' and
        row['score_cat'] == 'high'):
        return ["Pomodoro", "Active Recall", "Blurting Method"]

    # RULE 2
    elif (row['sleep_cat'] == 'low' and
          row['hours_cat'] == 'low' and
          row['motivation_cat'] == 'low'):
        return ["Pomodoro", "Interleaving", "Mind Mapping"]

    # RULE 3
    elif (row['hours_cat'] == 'high' and
          row['motivation_cat'] == 'high' and
          row['score_cat'] == 'medium'):
        return ["Feynman Technique", "Active Recall", "Blurting Method"]

    # RULE 4
    elif (row['motivation_cat'] == 'low' and
          row['parental_cat'] == 'low' and
          row['tutoring_cat'] == 'high'):
        return ["Interleaving", "Pomodoro", "Mind Mapping"]

    # RULE 5
    elif (row['resource_cat'] == 'low' and
          row['tutoring_cat'] == 'high'):
        return ["SQ3R", "Feynman Technique", "Teach Back"]

    # RULE 6
    elif (row['sleep_cat'] == 'high' and
          row['motivation_cat'] == 'high' and
          row['hours_cat'] == 'high'):
        return ["Active Recall", "Spaced Repetition", "Feynman Technique"]

    # RULE 7
    elif (row['disability_cat'] == 'high' and
          row['hours_cat'] == 'low'):
        return ["Mind Mapping", "Teach Back", "Feynman Technique"]

    # RULE 8
    elif (row['score_cat'] == 'low' and
          row['motivation_cat'] == 'medium' and
          row['hours_cat'] == 'medium'):
        return ["Spaced Repetition", "Active Recall", "Blurting Method"]

    # RULE 9
    elif (row['tutoring_cat'] == 'low' and
          row['motivation_cat'] == 'medium'):
        return ["Active Recall", "Blurting Method", "Feynman Technique"]

    # RULE 10
    elif (row['parental_cat'] == 'high' and
          row['motivation_cat'] == 'low'):
        return ["Pomodoro", "Interleaving", "Teach Back"]

    # RULE 11
    elif (row['sleep_cat'] == 'low' and
          row['disability_cat'] == 'medium'):
        return ["Pomodoro", "Mind Mapping", "Spaced Repetition"]

    # RULE 12
    elif (row['hours_cat'] == 'low' and
          row['motivation_cat'] == 'medium' and
          row['resource_cat'] == 'high'):
        return ["Pomodoro", "Active Recall", "Interleaving"]

    # DEFAULT (jika tidak ada rule cocok)
    return ["Active Recall", "Pomodoro"]


# APPLY (tanpa warning)
df.loc[:, 'rekomendasi'] = df.apply(rekomendasi, axis=1)


# OUTPUT HASIL
print("\n=== HASIL REKOMENDASI (FORWARD CHAINING) ===")
print(df[[
    'hours_cat',
    'score_cat',
    'sleep_cat',
    'motivation_cat',
    'parental_cat',
    'resource_cat',
    'tutoring_cat',
    'disability_cat',
    'rekomendasi'
]].head())


=== HASIL REKOMENDASI (FORWARD CHAINING) ===
  hours_cat score_cat sleep_cat motivation_cat parental_cat resource_cat  \
0       low    medium    medium            low          low         high   
1       low      high    medium            low          low       medium   
2       low       low    medium         medium       medium       medium   
3       low       low    medium         medium          low       medium   
4       low    medium    medium         medium       medium       medium   

  tutoring_cat disability_cat                                    rekomendasi  
0         high           high         [Interleaving, Pomodoro, Mind Mapping]  
1         high           high     [Pomodoro, Active Recall, Blurting Method]  
2         high           high  [Mind Mapping, Teach Back, Feynman Technique]  
3         high           high  [Mind Mapping, Teach Back, Feynman Technique]  
4         high           high  [Mind Mapping, Teach Back, Feynman Technique]  
